In [1]:
# === Manual Biphase Correction Tool ===

import os
import numpy as np
import tifffile
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output


In [4]:
# ==== USER INPUT ====
input_root = 'z:\\2025\\mbo_lbm_proccessed\\mk301\\mk301_02_27_2025\\red_registered_before_only_processed'
file_suffix = '_frames_0_to_1000_mean_autocontrast.tiff'


In [ ]:
# ==== STATE ====
plane_folders = sorted([f for f in os.listdir(input_root) if f.startswith('plane_')])
current_plane_index = 0
current_image = None
current_path = None

# ==== WIDGETS ====
even_shift = widgets.IntSlider(value=0, min=-20, max=20, description="Even shift")
odd_shift = widgets.IntSlider(value=0, min=-20, max=20, description="Odd shift")
save_button = widgets.Button(description="Save & Next Plane", button_style='success')
output = widgets.Output()

# ==== FUNCTIONS ====

def load_image(path):
    return tifffile.imread(path)

def apply_line_shift(img, even_val, odd_val):
    shifted = np.copy(img)
    shifted[0::2] = np.roll(shifted[0::2], even_val, axis=1)
    shifted[1::2] = np.roll(shifted[1::2], odd_val, axis=1)
    return shifted

def update_display():
    with output:
        clear_output(wait=True)
        shifted = apply_line_shift(current_image, even_shift.value, odd_shift.value)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(shifted, cmap='gray')
        ax.set_title(f"{plane_folders[current_plane_index]}")
        ax.axis('off')
        plt.show()

def load_next_image():
    global current_image, current_path
    if current_plane_index >= len(plane_folders):
        with output:
            clear_output(wait=True)
            print("✅ All planes processed!")
        return

    folder = plane_folders[current_plane_index]
    current_path = os.path.join(input_root, folder, f"{folder}{file_suffix}")

    if os.path.exists(current_path):
        current_image = load_image(current_path)
        even_shift.value = 0
        odd_shift.value = 0
        update_display()
    else:
        with output:
            clear_output(wait=True)
            print(f"⚠️ Image not found for {folder}. Skipping...")
        next_plane(None)  # Skip if missing file

def next_plane(b):
    global current_plane_index
    # Save current plane
    shifted = apply_line_shift(current_image, even_shift.value, odd_shift.value)

    # Overwrite TIFF
    tifffile.imwrite(current_path, shifted.astype(np.uint16))

    # Also overwrite PNG
    png_path = current_path.replace('.tiff', '.png')
    plt.imsave(png_path, shifted, cmap='gray')

    # Move to next plane
    current_plane_index += 1
    load_next_image()

def on_slider_change(change):
    update_display()

# ==== CONNECT EVENTS ====
even_shift.observe(on_slider_change, names='value')
odd_shift.observe(on_slider_change, names='value')
save_button.on_click(next_plane)

# ==== LAUNCH ====
load_next_image()
display(widgets.VBox([
    widgets.HBox([even_shift, odd_shift, save_button]),
    output
]))